# Melton Name Enrichment Exploration

## Overview

This notebook explores generated-name Vicmap locations within the Melton LGA
for Iteration 2.

The main source considered is the official Melton City Council spatial data,
including parks, reserves and recreation facilities. The exploration will:

1. identify the Melton locations that still use generated names;
2. inspect the official source structure and name coverage;
3. perform spatial matching;
4. review uncertain candidates; and
5. record which official names are suitable for later enrichment.

This notebook performs exploration only. It does not overwrite the Iteration 1
dataset or the final application CSV.

In [12]:
from pathlib import Path

import pandas as pd
import geopandas as gpd

# Locate the repository root.
def find_project_root(start: Path) -> Path:
    """Find the folder containing data and pipeline."""

    for folder in [start.resolve(), *start.resolve().parents]:
        if (folder / "data").is_dir() and (folder / "pipeline").is_dir():
            return folder

    raise FileNotFoundError("Could not locate the project root.")


PROJECT_ROOT = find_project_root(Path.cwd())

PLACES_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "vicmap"
    / "vicmap_app_ready.csv"
)

# Load the Iteration 1 output.
places = pd.read_csv(
    PLACES_PATH,
    encoding="utf-8-sig",
)

print("Rows:", f"{len(places):,}")
print("Columns:", len(places.columns))

Rows: 3,237
Columns: 14


In [13]:
# Select Melton records that still use generated names.
melton_mask = (
    places["lga_name"]
    .astype("string")
    .str.strip()
    .str.upper()
    .eq("MELTON")
    & places["name_source"]
    .astype("string")
    .str.strip()
    .eq("generated_from_subtype")
)

melton_unnamed = places.loc[melton_mask].copy()

# Convert target records into spatial points.
melton_unnamed_gdf = gpd.GeoDataFrame(
    melton_unnamed.copy(),
    geometry=gpd.points_from_xy(
        melton_unnamed["longitude"],
        melton_unnamed["latitude"],
    ),
    crs="EPSG:4326",
)

print("Melton generated-name locations:", len(melton_unnamed))

display(
    melton_unnamed[
        [
            "place_id",
            "display_name",
            "activity_category",
            "feature_subtype",
            "longitude",
            "latitude",
        ]
    ].head()
)

Melton generated-name locations: 111


,place_id,display_name,activity_category,feature_subtype,longitude,latitude
623,vicmap_foi_1010719,Unnamed Basketball Court - Melton - 1010719,court,basketball court,144.736681,-37.685617
624,vicmap_foi_1167975,Unnamed Basketball Court - Melton - 1167975,court,basketball court,144.745565,-37.792392
625,vicmap_foi_1167980,Unnamed Basketball Court - Melton - 1167980,court,basketball court,144.744473,-37.782537
626,vicmap_foi_1168000,Unnamed Basketball Court - Melton - 1168000,court,basketball court,144.747402,-37.791403
627,vicmap_foi_1168010,Unnamed Basketball Court - Melton - 1168010,court,basketball court,144.746911,-37.791548


In [14]:
# Summarise the targets by category and subtype.
melton_target_summary = (
    melton_unnamed
    .groupby(
        ["activity_category", "feature_subtype"]
    )
    .size()
    .reset_index(name="locations")
)

display(melton_target_summary)

# Confirm the target population is usable.
assert len(melton_unnamed) == 111
assert melton_unnamed["place_id"].is_unique
assert melton_unnamed[
    ["longitude", "latitude"]
].notna().all().all()

print("Melton target checks passed.")

,activity_category,feature_subtype,locations
0,court,basketball court,10
1,court,netball court,7
2,court,tennis court,3
3,park_and_garden,park,52
4,sports_ground,hockey ground,1
5,sports_ground,sports complex,2
6,sports_ground,sports ground,36


Melton target checks passed.


### Target population summary

The Iteration 1 dataset contains **111 Melton locations** that still use
generated names. All target records have unique place IDs and complete
coordinates.

The targets consist of:

- 20 courts;
- 52 parks and gardens; and
- 39 sports grounds or sports complexes.

Parks are the largest target group, followed by general sports grounds. The
target data passed the initial checks and is ready for comparison with the
official Melton City Council spatial data.

In [15]:
# Official Melton Open Space GeoJSON.
MELTON_OPEN_SPACE_URL = (
    "https://data.gov.au/geoserver/melton-open-space/wfs"
    "?request=GetFeature"
    "&typeName=ckan_e3fb7007_d697_4c92_b3bb_4420478e5e1f"
    "&outputFormat=json"
)

# Load the official council dataset.
melton_open_space = gpd.read_file(MELTON_OPEN_SPACE_URL)

# Correct the incorrect CRS label, then convert to latitude/longitude.
melton_open_space = (
    melton_open_space
    .set_crs("EPSG:28355", allow_override=True)
    .to_crs("EPSG:4326")
)

print("Official open-space records:", len(melton_open_space))
print("CRS:", melton_open_space.crs)
print("Bounds:", melton_open_space.total_bounds)

display(
    melton_open_space[
        ["Asset_Id", "Asset_Name", "Asset_Type", "Asset_Subt", "Locality"]
    ].head()
)

Official open-space records: 1347
CRS: EPSG:4326
Bounds: [144.51769066 -37.79886334 144.76766136 -37.56928198]


,Asset_Id,Asset_Name,Asset_Type,Asset_Subt,Locality
0,351,Arthur Westlake Recreation Reserve,Active Recreation,Structured Sporting,Melton West
1,49931,Western BACE Surrounds,Passive Recreation,Facilityscape,Cobblebank
2,2020981,Lunar Way Reserve,Passive Recreation,Social Recreation,Fraser Rise
3,2021566,Auburn Dr Reserve,Passive Recreation,Social Recreation,Fraser Rise
4,2021547,Argus Cl Courthead,Passive Recreation,Linear and Walkways,Diggers Rest


In [16]:
# Keep named open-space polygons only.
named_open_space = melton_open_space[
    melton_open_space["Asset_Name"].notna()
    & melton_open_space["Asset_Name"].str.strip().ne("")
].copy()

# Match target points to containing open spaces.
open_space_matches = gpd.sjoin(
    melton_unnamed_gdf,
    named_open_space[
        [
            "Asset_Id",
            "Asset_Name",
            "Asset_Type",
            "Asset_Subt",
            "Locality",
            "geometry",
        ]
    ],
    how="left",
    predicate="within",
)

# Report initial coverage.
matched_mask = open_space_matches["Asset_Name"].notna()

print("Targets inside a named open space:", matched_mask.sum())
print("Unmatched targets:", (~matched_mask).sum())
print(
    "Targets with multiple matches:",
    open_space_matches.index.duplicated(keep=False).sum(),
)

Targets inside a named open space: 2
Unmatched targets: 109
Targets with multiple matches: 0


In [17]:
# Project both datasets into a metric CRS.
targets_m = melton_unnamed_gdf.to_crs("EPSG:7855")
open_space_m = named_open_space.to_crs("EPSG:7855")

# Find the nearest named open space for each target.
nearest_open_space = gpd.sjoin_nearest(
    targets_m,
    open_space_m[
        [
            "Asset_Id",
            "Asset_Name",
            "Asset_Type",
            "Asset_Subt",
            "Locality",
            "geometry",
        ]
    ],
    how="left",
    distance_col="open_space_distance_m",
)

# Keep one nearest result per target.
nearest_open_space = (
    nearest_open_space
    .sort_values(["place_id", "open_space_distance_m"])
    .drop_duplicates(subset="place_id")
)

# Compare several review distances.
distance_summary = pd.DataFrame(
    {
        "maximum_distance_m": [25, 50, 100, 200],
        "matched_locations": [
            nearest_open_space["open_space_distance_m"].le(distance).sum()
            for distance in [25, 50, 100, 200]
        ],
    }
)

distance_summary["unmatched_locations"] = (
    len(melton_unnamed_gdf)
    - distance_summary["matched_locations"]
)

display(distance_summary)

,maximum_distance_m,matched_locations,unmatched_locations
0,25,6,105
1,50,17,94
2,100,46,65
3,200,73,38


In [18]:
# Assign conservative distance-based review levels.
nearest_open_space["review_status"] = pd.cut(
    nearest_open_space["open_space_distance_m"],
    bins=[-1, 50, 100, float("inf")],
    labels=[
        "strong_candidate",
        "manual_review",
        "too_far",
    ],
)

# Summarise candidates by activity category.
candidate_summary = pd.crosstab(
    nearest_open_space["activity_category"],
    nearest_open_space["review_status"],
)

display(candidate_summary)

# Display candidates within 100 metres.
open_space_candidates = nearest_open_space[
    nearest_open_space["open_space_distance_m"].le(100)
].copy()

display(
    open_space_candidates[
        [
            "place_id",
            "activity_category",
            "feature_subtype",
            "display_name",
            "Asset_Name",
            "Asset_Type",
            "Asset_Subt",
            "open_space_distance_m",
            "review_status",
        ]
    ].sort_values(
        ["review_status", "open_space_distance_m"]
    )
)

review_status,strong_candidate,manual_review,too_far
activity_category,,,
court,4,7,9
park_and_garden,5,10,37
sports_ground,8,12,19


,place_id,activity_category,feature_subtype,display_name,Asset_Name,Asset_Type,Asset_Subt,open_space_distance_m,review_status
2477,vicmap_foi_1252007,sports_ground,sports ground,Unnamed Sports Ground - Melton - 1252007,Kirkton Park,Passive Recreation,Social Recreation,0.000000,strong_candidate
2466,vicmap_foi_75689,sports_ground,sports complex,Unnamed Sports Complex - Melton - 75689,MacPherson Park,Active Recreation,Structured Sporting,0.000000,strong_candidate
638,vicmap_foi_1252094,court,netball court,Unnamed Netball Court - Melton - 1252094,Blackwood Drive Recreation Reserve,Active Recreation,Structured Sporting,13.117887,strong_candidate
1940,vicmap_foi_1340990,park_and_garden,park,Unnamed Park - Melton - 1340990,Horsley St Reserve,Passive Recreation,Social Recreation,15.001942,strong_candidate
641,vicmap_foi_1313706,court,tennis court,Unnamed Tennis Court - Melton - 1313706,Botanica Springs Linear Reserve,Passive Recreation,Linear and Walkways,19.945660,strong_candidate
2489,vicmap_foi_1252098,sports_ground,sports ground,Unnamed Sports Ground - Melton - 1252098,Lachlan Lane Closure,Passive Recreation,Streetscape,22.131687,strong_candidate
2484,vicmap_foi_1252066,sports_ground,sports ground,Unnamed Sports Ground - Melton - 1252066,Arthur Westlake Recreation Reserve,Active Recreation,Structured Sporting,27.680253,strong_candidate
623,vicmap_foi_1010719,court,basketball court,Unnamed Basketball Court - Melton - 1010719,Hillside Recreation Reserve,Active Recreation,Structured Sporting,30.960198,strong_candidate
1916,vicmap_foi_1335077,park_and_garden,park,Unnamed Park - Melton - 1335077,Lawler Rd Linear Reserve,Passive Recreation,Linear and Walkways,40.328331,strong_candidate
1949,vicmap_foi_638764,park_and_garden,park,Unnamed Park - Melton - 638764,Arnolds Creek South of Brooklyn Road Environme...,Environmental Protection,Environmental,40.835043,strong_candidate


In [19]:
# Inspect only candidates within 50 metres.
strong_candidates = nearest_open_space[
    nearest_open_space["review_status"].eq("strong_candidate")
].copy()

display(
    strong_candidates[
        [
            "place_id",
            "activity_category",
            "feature_subtype",
            "display_name",
            "Asset_Name",
            "Asset_Type",
            "Asset_Subt",
            "open_space_distance_m",
        ]
    ].sort_values("open_space_distance_m")
)

,place_id,activity_category,feature_subtype,display_name,Asset_Name,Asset_Type,Asset_Subt,open_space_distance_m
2477,vicmap_foi_1252007,sports_ground,sports ground,Unnamed Sports Ground - Melton - 1252007,Kirkton Park,Passive Recreation,Social Recreation,0.000000
2466,vicmap_foi_75689,sports_ground,sports complex,Unnamed Sports Complex - Melton - 75689,MacPherson Park,Active Recreation,Structured Sporting,0.000000
638,vicmap_foi_1252094,court,netball court,Unnamed Netball Court - Melton - 1252094,Blackwood Drive Recreation Reserve,Active Recreation,Structured Sporting,13.117887
1940,vicmap_foi_1340990,park_and_garden,park,Unnamed Park - Melton - 1340990,Horsley St Reserve,Passive Recreation,Social Recreation,15.001942
641,vicmap_foi_1313706,court,tennis court,Unnamed Tennis Court - Melton - 1313706,Botanica Springs Linear Reserve,Passive Recreation,Linear and Walkways,19.945660
2489,vicmap_foi_1252098,sports_ground,sports ground,Unnamed Sports Ground - Melton - 1252098,Lachlan Lane Closure,Passive Recreation,Streetscape,22.131687
2484,vicmap_foi_1252066,sports_ground,sports ground,Unnamed Sports Ground - Melton - 1252066,Arthur Westlake Recreation Reserve,Active Recreation,Structured Sporting,27.680253
623,vicmap_foi_1010719,court,basketball court,Unnamed Basketball Court - Melton - 1010719,Hillside Recreation Reserve,Active Recreation,Structured Sporting,30.960198
1916,vicmap_foi_1335077,park_and_garden,park,Unnamed Park - Melton - 1335077,Lawler Rd Linear Reserve,Passive Recreation,Linear and Walkways,40.328331
1949,vicmap_foi_638764,park_and_garden,park,Unnamed Park - Melton - 638764,Arnolds Creek South of Brooklyn Road Environme...,Environmental Protection,Environmental,40.835043


In [20]:
# Show target groups associated with each official place.
display(
    strong_candidates
    .groupby(
        ["Asset_Name", "activity_category", "feature_subtype"],
        dropna=False,
    )
    .size()
    .reset_index(name="target_locations")
    .sort_values(
        ["Asset_Name", "target_locations"],
        ascending=[True, False],
    )
)

,Asset_Name,activity_category,feature_subtype,target_locations
0,Arnolds Creek Linear Reserve West Branch - Wes...,sports_ground,sports ground,1
1,Arnolds Creek South of Brooklyn Road Environme...,park_and_garden,park,1
2,Arthur Westlake Recreation Reserve,sports_ground,sports ground,1
3,Blackwood Drive Recreation Reserve,court,netball court,1
4,Blackwood Drive Recreation Reserve,sports_ground,sports ground,1
5,Botanica Springs Linear Reserve,court,tennis court,1
6,Diggers Rest Recreation Reserve,sports_ground,sports ground,1
7,Hillside Recreation Reserve,court,basketball court,1
8,Horsley St Reserve,park_and_garden,park,1
9,Kirkton Park,sports_ground,sports ground,1


In [21]:
# Record the manually confirmed open-space matches.
accepted_open_space = strong_candidates.copy()
accepted_open_space["review_status"] = "accepted"
accepted_open_space["review_note"] = (
    "Official open-space match within 50 m; manually confirmed."
)

# Keep unresolved targets for the next source.
remaining_targets = melton_unnamed_gdf[
    ~melton_unnamed_gdf["place_id"].isin(
        accepted_open_space["place_id"]
    )
].copy()

print("Accepted open-space names:", len(accepted_open_space))
print("Remaining targets:", len(remaining_targets))

Accepted open-space names: 17
Remaining targets: 94


### Open Space Match Review

The official Melton Open Space dataset was compared with the 111 generated-name targets. A conservative 50-metre threshold identified 17 strong spatial candidates.

These 17 candidates were manually reviewed and accepted because the official open-space names were consistent with their corresponding Vicmap locations. The remaining 94 targets were retained for assessment against other official Melton datasets. No names were updated based on distance alone.

In [22]:
# Official Melton Recreation Facilities GeoJSON.
MELTON_RECREATION_URL = (
    "https://data.gov.au/geoserver/melton-recreation-facilities/wfs"
    "?request=GetFeature"
    "&typeName=ckan_15a9b3b6_263c_4dbd_8f80_e98333cef48f"
    "&outputFormat=json"
)

# Load and correct the source CRS metadata.
melton_recreation = gpd.read_file(MELTON_RECREATION_URL)

melton_recreation = (
    melton_recreation
    .set_crs("EPSG:28355", allow_override=True)
    .to_crs("EPSG:4326")
)

print("Recreation facility records:", len(melton_recreation))
print("CRS:", melton_recreation.crs)
print("Bounds:", melton_recreation.total_bounds)

display(
    melton_recreation[
        [
            "Asset_id",
            "Asset_Name",
            "Asset_Loca",
            "Asset_Subt",
        ]
    ].head()
)

Recreation facility records: 67
CRS: EPSG:4326
Bounds: [144.54373218 -37.79356559 144.75561346 -37.60805379]


,Asset_id,Asset_Name,Asset_Loca,Asset_Subt
0,46537,Brookside Stadium,"13-16 Caroline Springs BVD, CAROLINE SPRINGS 3023",Stadium
1,41,Parkwood Green - Tennis Pavilion,"88-94 Catherine DR, HILLSIDE 3037",Pavilion/Clubrooms
2,155,Melton Pistol Club - Enclosed Shooting Range,"362 Clarkes RD, BROOKFIELD 3338",Stadium
3,80,Mt Carberry Cricket Pavilion (DJ Cunningham),"41 Exford RD, MELTON SOUTH 3338",Pavilion/Clubrooms
4,71,Blackwood Drive Scout Hall,"2-20 Reynolds PL, MELTON SOUTH 3338",Scouts Hall


In [23]:
# Select unresolved sport-related targets only.
recreation_targets = remaining_targets[
    remaining_targets["activity_category"].isin(
        ["court", "sports_ground"]
    )
].copy()

# Keep official facilities with usable names.
named_recreation = melton_recreation[
    melton_recreation["Asset_Name"].notna()
    & melton_recreation["Asset_Name"].str.strip().ne("")
].copy()

# Convert both datasets to a metric CRS.
recreation_targets_m = recreation_targets.to_crs("EPSG:7855")
named_recreation_m = named_recreation.to_crs("EPSG:7855")

# Find the nearest official recreation facility.
nearest_recreation = gpd.sjoin_nearest(
    recreation_targets_m,
    named_recreation_m[
        [
            "Asset_id",
            "Asset_Name",
            "Asset_Loca",
            "Asset_Subt",
            "geometry",
        ]
    ],
    how="left",
    distance_col="recreation_distance_m",
)

# Keep one nearest facility per target.
nearest_recreation = (
    nearest_recreation
    .sort_values(["place_id", "recreation_distance_m"])
    .drop_duplicates(subset="place_id")
)

print("Sport-related targets:", len(recreation_targets))
print("Named recreation facilities:", len(named_recreation))

Sport-related targets: 47
Named recreation facilities: 67


In [24]:
# Compare conservative matching distances.
recreation_distance_summary = pd.DataFrame(
    {
        "maximum_distance_m": [25, 50, 100, 200],
        "matched_locations": [
            nearest_recreation["recreation_distance_m"]
            .le(distance)
            .sum()
            for distance in [25, 50, 100, 200]
        ],
    }
)

recreation_distance_summary["unmatched_locations"] = (
    len(recreation_targets)
    - recreation_distance_summary["matched_locations"]
)

display(recreation_distance_summary)

,maximum_distance_m,matched_locations,unmatched_locations
0,25,0,47
1,50,0,47
2,100,0,47
3,200,5,42


### Recreation Facilities Match Assessment

The general Melton Recreation Facilities dataset was tested against the 47 unresolved court and sports-ground targets.

No targets were located within 100 metres of an official facility, while only five were found within 200 metres. This distance was considered too large for reliable name assignment. Therefore, no names were accepted from this dataset, and more specific facility datasets were explored instead.

In [26]:
# Official specialised facility sources.
SPECIALISED_URLS = {
    "ovals": (
        "https://data.gov.au/geoserver/melton-ovals-and-fields/wfs"
        "?request=GetFeature"
        "&typeName=ckan_91517082_af13_4663_84ce_604a92d24e86"
        "&outputFormat=json"
    ),
    "netball": (
        "https://data.gov.au/geoserver/melton-netball-courts/wfs"
        "?request=GetFeature"
        "&typeName=ckan_b7541fe3_95e8_4ec9_b095_1f411bc87753"
        "&outputFormat=json"
    ),
    "tennis": (
        "https://data.gov.au/geoserver/melton-tennis-courts/wfs"
        "?request=GetFeature"
        "&typeName=ckan_bcb69750_47e8_4ea9_ade3_42cb484289cc"
        "&outputFormat=json"
    ),
}

# Load and standardise the official layers.
specialised_layers = []

for source_name, source_url in SPECIALISED_URLS.items():
    layer = gpd.read_file(source_url)

    # Correct the source CRS metadata.
    layer = (
        layer
        .set_crs("EPSG:28355", allow_override=True)
        .to_crs("EPSG:4326")
    )

    layer["official_source"] = source_name
    specialised_layers.append(layer)

specialised_facilities = gpd.GeoDataFrame(
    pd.concat(specialised_layers, ignore_index=True),
    geometry="geometry",
    crs="EPSG:4326",
)

display(
    specialised_facilities[
        ["official_source", "Asset_Name"]
    ]
    .groupby("official_source")
    .count()
    .rename(columns={"Asset_Name": "records"})
)

,records
official_source,
netball,16
ovals,46
tennis,70


In [27]:
# Pair each official layer with relevant target types.
target_source_pairs = [
    (
        "ovals",
        remaining_targets[
            remaining_targets["activity_category"].eq("sports_ground")
        ],
    ),
    (
        "netball",
        remaining_targets[
            remaining_targets["feature_subtype"].eq("netball court")
        ],
    ),
    (
        "tennis",
        remaining_targets[
            remaining_targets["feature_subtype"].eq("tennis court")
        ],
    ),
]

specialised_match_results = []

for source_name, target_group in target_source_pairs:
    source_layer = specialised_facilities[
        specialised_facilities["official_source"].eq(source_name)
    ]

    # Match target points inside the relevant facility polygons.
    result = gpd.sjoin(
        target_group,
        source_layer[
            [
                "Asset_Id",
                "Asset_Name",
                "Asset_Type",
                "Asset_Subt",
                "Locality",
                "official_source",
                "geometry",
            ]
        ],
        how="left",
        predicate="within",
    )

    specialised_match_results.append(result)

specialised_matches = pd.concat(
    specialised_match_results,
    ignore_index=True,
)

# Count unique tested and matched targets.
tested_ids = set(specialised_matches["place_id"])
matched_ids = set(
    specialised_matches.loc[
        specialised_matches["Asset_Name"].notna(),
        "place_id",
    ]
)

print("Specialised targets tested:", len(tested_ids))
print("Exact specialised matches:", len(matched_ids))
print("Unmatched specialised targets:", len(tested_ids - matched_ids))

Specialised targets tested: 38
Exact specialised matches: 0
Unmatched specialised targets: 38


In [29]:
# Find the nearest relevant facility for each target group.
nearest_specialised_results = []

for source_name, target_group in target_source_pairs:
    source_layer = specialised_facilities[
        specialised_facilities["official_source"].eq(source_name)
    ]

    # Use a metric CRS for distance calculation.
    target_group_m = target_group.to_crs("EPSG:7855")
    source_layer_m = source_layer.to_crs("EPSG:7855")

    result = gpd.sjoin_nearest(
        target_group_m,
        source_layer_m[
            [
                "Asset_Id",
                "Asset_Name",
                "Asset_Type",
                "Asset_Subt",
                "Locality",
                "official_source",
                "geometry",
            ]
        ],
        how="left",
        distance_col="facility_distance_m",
    )

    # Keep one nearest result per target.
    result = (
        result
        .sort_values(["place_id", "facility_distance_m"])
        .drop_duplicates(subset="place_id")
    )

    nearest_specialised_results.append(result)

nearest_specialised = pd.concat(
    nearest_specialised_results,
    ignore_index=True,
)

In [30]:
# Summarise distance coverage by source.
distance_rows = []

for source_name in ["ovals", "netball", "tennis"]:
    source_results = nearest_specialised[
        nearest_specialised["official_source"].eq(source_name)
    ]

    for distance in [25, 50, 100, 200]:
        distance_rows.append(
            {
                "official_source": source_name,
                "maximum_distance_m": distance,
                "matched_locations": (
                    source_results["facility_distance_m"]
                    .le(distance)
                    .sum()
                ),
                "total_targets": len(source_results),
            }
        )

specialised_distance_summary = pd.DataFrame(distance_rows)
display(specialised_distance_summary)

,official_source,maximum_distance_m,matched_locations,total_targets
0,ovals,25,0,31
1,ovals,50,0,31
2,ovals,100,2,31
3,ovals,200,8,31
4,netball,25,0,5
5,netball,50,0,5
6,netball,100,0,5
7,netball,200,0,5
8,tennis,25,0,2
9,tennis,50,0,2


In [31]:
# Keep only oval candidates within 100 metres.
oval_review = nearest_specialised[
    nearest_specialised["official_source"].eq("ovals")
    & nearest_specialised["facility_distance_m"].le(100)
].copy()

oval_review["review_status"] = "manual_review"

display(
    oval_review[
        [
            "place_id",
            "activity_category",
            "feature_subtype",
            "display_name",
            "Asset_Name",
            "Asset_Type",
            "Locality",
            "facility_distance_m",
            "longitude",
            "latitude",
            "review_status",
        ]
    ].sort_values("facility_distance_m")
)

,place_id,activity_category,feature_subtype,display_name,Asset_Name,Asset_Type,Locality,facility_distance_m,longitude,latitude,review_status
27,vicmap_foi_75891,sports_ground,sports ground,Unnamed Sports Ground - Melton - 75891,Kurunjang Recreation Reserve - Cricket/Footbal...,Cricket/Football Oval,Kurunjang,78.480616,144.583929,-37.671263,manual_review
18,vicmap_foi_1275014,sports_ground,sports ground,Unnamed Sports Ground - Melton - 1275014,Eynesbury Recreation Reserve - Cricket/Footbal...,Cricket/Football Oval,Eynesbury,93.035580,144.546677,-37.790883,manual_review


In [32]:
# Store the two manual review decisions.
oval_review = (
    oval_review
    .sort_values("facility_distance_m")
    .copy()
)

oval_review["review_status"] = "rejected"
oval_review["review_note"] = (
    "Nearby official oval, but the target location was not confirmed."
)

# Accept the first row confirmed as a football ground.
first_match_index = oval_review.index[0]

oval_review.loc[
    first_match_index,
    "review_status",
] = "accepted"

oval_review.loc[
    first_match_index,
    "review_note",
] = "Official football ground manually confirmed."

display(
    oval_review[
        [
            "place_id",
            "display_name",
            "Asset_Name",
            "facility_distance_m",
            "review_status",
            "review_note",
        ]
    ]
)

,place_id,display_name,Asset_Name,facility_distance_m,review_status,review_note
27,vicmap_foi_75891,Unnamed Sports Ground - Melton - 75891,Kurunjang Recreation Reserve - Cricket/Footbal...,78.480616,accepted,Official football ground manually confirmed.
18,vicmap_foi_1275014,Unnamed Sports Ground - Melton - 1275014,Eynesbury Recreation Reserve - Cricket/Footbal...,93.035580,rejected,"Nearby official oval, but the target location ..."


In [33]:
# Standardise accepted open-space matches.
accepted_open_space_final = accepted_open_space.assign(
    official_name=accepted_open_space["Asset_Name"],
    source_dataset="Melton Open Space",
    match_distance_m=accepted_open_space["open_space_distance_m"],
)[
    [
        "place_id",
        "official_name",
        "source_dataset",
        "match_distance_m",
        "review_status",
        "review_note",
    ]
]

# Keep the single accepted oval match.
accepted_oval_final = oval_review[
    oval_review["review_status"].eq("accepted")
].assign(
    official_name=lambda frame: frame["Asset_Name"],
    source_dataset="Melton Ovals and Fields",
    match_distance_m=lambda frame: frame["facility_distance_m"],
)[
    [
        "place_id",
        "official_name",
        "source_dataset",
        "match_distance_m",
        "review_status",
        "review_note",
    ]
]

# Combine all accepted Melton name candidates.
accepted_melton_names = pd.concat(
    [
        accepted_open_space_final,
        accepted_oval_final,
    ],
    ignore_index=True,
)

# Identify records that retain generated names.
unresolved_melton = melton_unnamed[
    ~melton_unnamed["place_id"].isin(
        accepted_melton_names["place_id"]
    )
].copy()

assert accepted_melton_names["place_id"].is_unique
assert len(accepted_melton_names) == 18
assert len(unresolved_melton) == 93

print("Accepted official names:", len(accepted_melton_names))
print("Generated names retained:", len(unresolved_melton))
print("Total Melton targets:", len(melton_unnamed))

Accepted official names: 18
Generated names retained: 93
Total Melton targets: 111


## Exploration Conclusion

The exploration assessed 111 Melton locations that originally used generated names.

The official Melton Open Space dataset produced 17 candidates within 50 metres that were accepted after manual review. The specialised Ovals and Fields dataset contributed one additional manually confirmed match. In total, 18 official name candidates were accepted.

The general Recreation Facilities dataset produced no candidates within 100 metres. The specialised netball and tennis datasets produced no candidates within 200 metres, while the available basketball GeoJSON contained no records. These sources were therefore not used for name assignment.

The remaining 93 locations retain their generated names because no sufficiently reliable official match was identified. Wider distance thresholds were not adopted because they would increase the risk of assigning a nearby facility’s name to the wrong Vicmap feature.

This notebook records exploration and review decisions only. It does not overwrite the processed application dataset.